### Installation of Evaluation Dependencies

Additional Python libraries are installed to support image processing and quantitative evaluation of the segmentation results. **Pillow** provides image-processing functionality required for reading, manipulating, and analyzing image data, while **pycocotools** provides utilities for working with the COCO annotation format and computing standard object detection and segmentation evaluation metrics.

These dependencies support the subsequent stages of result analysis, particularly the conversion, processing, and quantitative assessment of predicted segmentation masks against the corresponding ground-truth annotations.


In [ ]:
!pip install pillow pycocotools

### Dataset Path Configuration

The dataset directory structure is defined programmatically to provide consistent access to the training, validation, and test subsets of the **Purple Kale** dataset. Separate paths are established for the corresponding images and annotation files within each subset.

An output directory is also initialized for storing the generated COCO-format files and other intermediate evaluation artifacts. The directory is created automatically if it does not already exist, ensuring that subsequent processing steps can write their outputs without requiring manual directory creation.

Finally, the configured paths for the training, validation, and test image directories are displayed to verify that the dataset locations have been correctly specified. This configuration establishes the file-system structure required for subsequent annotation conversion and segmentation evaluation.


In [ ]:
import os

DATASET_DIR = "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/all/purpleKale"

TRAIN_IMAGES = f"{DATASET_DIR}/train/images"
TRAIN_LABELS = f"{DATASET_DIR}/train/labels"

VAL_IMAGES = f"{DATASET_DIR}/valid/images"
VAL_LABELS = f"{DATASET_DIR}/valid/labels"

TEST_IMAGES = f"{DATASET_DIR}/test/images"
TEST_LABELS = f"{DATASET_DIR}/test/labels"

OUTPUT_DIR = "/kaggle/working/coco"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Train:", TRAIN_IMAGES)
print("Val:  ", VAL_IMAGES)
print("Test: ", TEST_IMAGES)

In [ ]:
for name, path in [
    ("Train images", TRAIN_IMAGES),
    ("Train labels", TRAIN_LABELS),
    ("Val images", VAL_IMAGES),
    ("Val labels", VAL_LABELS),
    ("Test images", TEST_IMAGES),
    ("Test labels", TEST_LABELS),
]:
    print(f"{name}: {len(os.listdir(path))}")

### YOLO-to-COCO Annotation Conversion

A conversion procedure is implemented to transform polygon-based annotations from the YOLO segmentation format into the **COCO instance segmentation format**. This conversion enables the annotated dataset to be processed using COCO-compatible evaluation tools and metrics.

The procedure first identifies all supported image files within the specified image directory and retrieves their dimensions using the Pillow library. Each image is assigned a unique identifier, while the corresponding annotation file is located using the image filename. Images without an associated annotation file are retained in the dataset metadata but do not contribute segmentation annotations.

For each valid YOLO annotation, the class identifier and normalized polygon coordinates are extracted. The normalized coordinates, whose values are expressed relative to the image dimensions, are converted into absolute pixel coordinates using the image width and height. The coordinates are additionally constrained to the valid image boundaries to prevent invalid polygon vertices.

From each polygon, an axis-aligned bounding box is derived by determining the minimum and maximum horizontal and vertical coordinates. The resulting bounding box is represented using the COCO convention of \([x, y, width, height]\). The polygon is also converted into a run-length representation using `pycocotools` to calculate its segmentation area.

Each annotation is assigned a unique identifier and associated with its corresponding image. Because COCO category identifiers conventionally begin at one, the YOLO class index is incremented by one when constructing the COCO `category_id`. The generated annotations contain the polygon segmentation, bounding box, calculated area, and crowd indicator.

Finally, the procedure constructs the complete COCO dataset structure, including image metadata, annotations, category definitions, and dataset information. The resulting structure is serialized as a JSON file, providing a standardized representation of the segmentation dataset suitable for subsequent quantitative evaluation.


In [ ]:
import json
from PIL import Image
from pycocotools import mask as mask_utils


def convert_yolo_to_coco(image_dir, label_dir, output_json, class_names):

    images = []
    annotations = []

    image_id = 1
    annotation_id = 1

    image_extensions = (
        ".jpg", ".jpeg", ".png",
        ".bmp", ".tif", ".tiff"
    )

    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith(image_extensions)
    ])

    print(f"Found {len(image_files)} images")

    for image_file in image_files:

        image_path = os.path.join(image_dir, image_file)

        with Image.open(image_path) as img:
            width, height = img.size

        images.append({
            "id": image_id,
            "file_name": image_file,
            "width": width,
            "height": height
        })

        label_file = os.path.splitext(image_file)[0] + ".txt"
        label_path = os.path.join(label_dir, label_file)

        if not os.path.exists(label_path):
            image_id += 1
            continue

        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:

            values = line.strip().split()

            # class + at least 3 (x,y) points
            if len(values) < 7:
                continue

            class_id = int(values[0])
            coords = list(map(float, values[1:]))

            if len(coords) % 2 != 0:
                print(f"Skipping malformed annotation: {label_path}")
                continue

            # YOLO normalized coordinates -> pixel coordinates
            polygon = []

            for i in range(0, len(coords), 2):

                x = coords[i] * width
                y = coords[i + 1] * height

                x = max(0, min(x, width))
                y = max(0, min(y, height))

                polygon.extend([x, y])

            xs = polygon[0::2]
            ys = polygon[1::2]

            x_min = min(xs)
            y_min = min(ys)
            x_max = max(xs)
            y_max = max(ys)

            bbox_width = x_max - x_min
            bbox_height = y_max - y_min

            # Polygon area
            rle = mask_utils.frPyObjects(
                [polygon],
                height,
                width
            )

            area = float(mask_utils.area(rle)[0])

            annotations.append({
                "id": annotation_id,
                "image_id": image_id,

                # COCO category IDs start at 1
                "category_id": class_id + 1,

                "segmentation": [polygon],

                "bbox": [
                    x_min,
                    y_min,
                    bbox_width,
                    bbox_height
                ],

                "area": area,
                "iscrowd": 0
            })

            annotation_id += 1

        image_id += 1

    categories = [
        {
            "id": i + 1,
            "name": name,
            "supercategory": "object"
        }
        for i, name in enumerate(class_names)
    ]

    coco = {
        "info": {
            "description": "Kale instance segmentation dataset"
        },
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": categories
    }

    with open(output_json, "w") as f:
        json.dump(coco, f)

    print(f"\nSaved: {output_json}")
    print(f"Images: {len(images)}")
    print(f"Annotations: {len(annotations)}")
    print(f"Categories: {len(categories)}")

### Training Annotation Conversion and Class Configuration

The training annotations are converted from the YOLO segmentation format into a COCO-compatible JSON representation using the previously defined conversion procedure. The output annotation file is stored in the designated COCO output directory and is used for subsequent evaluation and analysis.

The `CLASS_NAMES` configuration specifies the semantic category represented in the dataset. In the current experiment, the class is defined as **Purple Kale**, corresponding to the crop being analyzed. This class name is incorporated into the COCO category metadata during conversion.

For experiments involving different crops, the class name must be updated to match the corresponding crop category before executing the conversion procedure. For example, when processing a Yellow Onion or another crop dataset, the value in `CLASS_NAMES` should be replaced with the appropriate crop name. This ensures that the generated COCO annotations contain the correct category label and remain consistent with the underlying dataset.

The conversion function is then applied to the training image and label directories, producing the final COCO-format training annotation file.


In [ ]:
TRAIN_COCO = f"{OUTPUT_DIR}/instances_train.json"
CLASS_NAMES = ['Purple Kale']

print(CLASS_NAMES)
convert_yolo_to_coco(
    TRAIN_IMAGES,
    TRAIN_LABELS,
    TRAIN_COCO,
    CLASS_NAMES
)

In [ ]:
VAL_COCO = f"{OUTPUT_DIR}/instances_val.json"

convert_yolo_to_coco(
    VAL_IMAGES,
    VAL_LABELS,
    VAL_COCO,
    CLASS_NAMES
)

In [ ]:
TEST_COCO = f"{OUTPUT_DIR}/instances_test.json"

convert_yolo_to_coco(
    TEST_IMAGES,
    TEST_LABELS,
    TEST_COCO,
    CLASS_NAMES
)

In [ ]:
from pycocotools.coco import COCO

for name, path in [
    ("TRAIN", TRAIN_COCO),
    ("VAL", VAL_COCO),
    ("TEST", TEST_COCO)
]:

    coco = COCO(path)

    print(
        f"{name}: "
        f"{len(coco.imgs)} images | "
        f"{len(coco.anns)} annotations | "
        f"{len(coco.cats)} classes"
    )

    print("Categories:", coco.cats)
    print()

### Installation of Supporting Dependencies

The required software dependencies for the subsequent computer vision and evaluation pipeline are installed in the experimental environment. The package set includes **PyTorch** and **Torchvision** for deep learning and vision-related operations, while **OpenCV** provides image processing and computer vision functionality.

The **pycocotools** library supports COCO-format annotations and evaluation, whereas **fvcore** and **iopath** provide supporting utilities for model evaluation, file handling, and data management. **OmegaConf** and **Hydra-Core** facilitate structured configuration and experiment management, while **Cloudpickle** supports serialization of Python objects. **TensorBoard** is included for monitoring and visualizing training-related metrics and experiment logs.

The quiet installation option is used to minimize console output during package installation. Collectively, these dependencies establish the computational environment required for model execution, dataset processing, evaluation, configuration management, and experiment monitoring.


In [ ]:
!pip install -q \
    torch \
    torchvision \
    opencv-python \
    pycocotools \
    fvcore \
    iopath \
    omegaconf \
    hydra-core \
    cloudpickle \
    tensorboard

### Detectron2 Framework Installation

The **Detectron2** framework developed by Meta AI is installed directly from its official GitHub repository. Detectron2 provides a modular computer vision framework for object detection and instance segmentation, including implementations and utilities for training, inference, dataset registration, and evaluation.

Installing the framework directly from the repository ensures that the experimental environment has access to the required Detectron2 implementation and its associated components. This framework is subsequently used for COCO-compatible instance segmentation processing and evaluation within the experimental pipeline.


In [ ]:
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import torch
import detectron2

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA:", torch.version.cuda)
print("Detectron2:", detectron2.__version__)

### COCO Dataset Registration in Detectron2

The converted COCO-format annotations are registered with the Detectron2 framework to make the training, validation, and test datasets accessible through its dataset management system. Three dataset instances are defined corresponding to the **training**, **validation**, and **test** splits of the Purple Kale dataset.

Each registration associates a unique dataset identifier with its corresponding COCO annotation file and image directory. The annotation files provide the image metadata, category information, segmentation polygons, bounding boxes, and other annotation attributes, while the image directories contain the corresponding input images.

This registration process enables Detectron2 to automatically load and interpret the dataset using its standard COCO data handling utilities. Separating the datasets into distinct identifiers also ensures that training, validation, and testing can be performed independently throughout the experimental pipeline.


In [ ]:
from detectron2.data.datasets import register_coco_instances

register_coco_instances(
    "kale_train",
    {},
    "/kaggle/working/coco/instances_train.json",
    "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/all/purpleKale/train/images"
)

register_coco_instances(
    "kale_val",
    {},
    "/kaggle/working/coco/instances_val.json",
    "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/all/purpleKale/valid/images"
)

register_coco_instances(
    "kale_test",
    {},
    "/kaggle/working/coco/instances_test.json",
    "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/all/purpleKale/test/images"
)

print("Datasets registered.")

### Dataset Registration Verification

The registered datasets are inspected using Detectron2's `DatasetCatalog` and `MetadataCatalog` interfaces to verify that the training, validation, and test datasets have been loaded correctly.

For each dataset split, the procedure retrieves the corresponding dataset records and metadata, then reports the total number of images, the registered object classes, the file path of the first image, and the number of annotations associated with that image. This provides a basic structural validation of the dataset before model training or evaluation.

The verification step is particularly useful for identifying inconsistencies in dataset registration, class definitions, image paths, or annotation loading. Confirming these properties before subsequent experiments helps ensure that the Detectron2 pipeline is operating on the intended dataset and annotation structure.


In [ ]:
from detectron2.data import MetadataCatalog, DatasetCatalog

for dataset_name in [
    "kale_train",
    "kale_val",
    "kale_test"
]:
    dataset = DatasetCatalog.get(dataset_name)
    metadata = MetadataCatalog.get(dataset_name)

    print(f"\n{dataset_name}")
    print("Images:", len(dataset))
    print("Classes:", metadata.thing_classes)
    print("First image:", dataset[0]["file_name"])
    print("Annotations:", len(dataset[0]["annotations"]))

### Mask R-CNN Model Configuration

A **Mask R-CNN with a ResNet-50 backbone and Feature Pyramid Network (FPN)** is configured using the Detectron2 framework. The model configuration is initialized from Detectron2's standard COCO instance-segmentation configuration, while the corresponding model weights are initialized using COCO-pretrained parameters. This transfer-learning strategy provides a pretrained visual representation that can be adapted to the target crop segmentation task.

The registered Purple Kale training dataset is assigned as the training set, while the validation dataset is used for periodic model evaluation. Two worker processes are allocated for data loading to support efficient dataset processing.

Since the target dataset contains a single object category, the number of ROI classification classes is set to one. The training configuration uses a batch size of **2 images per iteration**, a base learning rate of **0.00025**, and a maximum of **3,000 training iterations**. Model performance is evaluated on the validation dataset every **250 iterations**, allowing the progression of training to be monitored throughout optimization.

The experiment outputs, including model checkpoints and evaluation results, are stored in a dedicated output directory. The directory is created automatically if it does not already exist. This configuration therefore establishes the complete training environment for adapting the COCO-pretrained Mask R-CNN model to the Purple Kale instance segmentation task.


In [ ]:
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo

cfg = get_cfg()

# Mask R-CNN R50-FPN
cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
    )
)

# Dataset
cfg.DATASETS.TRAIN = ("kale_train",)
cfg.DATASETS.TEST = ("kale_val",)

# No separate test-time dataset during training
cfg.DATALOADER.NUM_WORKERS = 2

# COCO pretrained weights
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
)

# One class: kale
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1

# Training
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 3000

# Evaluate periodically
cfg.TEST.EVAL_PERIOD = 250

# Output
cfg.OUTPUT_DIR = "/kaggle/working/mask_rcnn_kale"

import os
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print(cfg)

### Custom Trainer and COCO Evaluator Configuration

A custom Detectron2 trainer, `KaleTrainer`, is defined by extending the framework's `DefaultTrainer` class. The purpose of this customization is to configure the evaluation procedure specifically for the registered COCO-format dataset.

The trainer overrides the `build_evaluator` method to instantiate a **COCOEvaluator** for the selected dataset. If an output directory is not explicitly provided, an `inference` subdirectory is automatically created within the configured experiment output directory. This ensures that evaluation results are stored separately from other training artifacts.

The `COCOEvaluator` is associated with the corresponding dataset name and configured to write its evaluation outputs to the designated directory. This enables standard COCO-compatible evaluation of the instance segmentation results.

At this stage, the custom trainer and evaluator are **only defined and configured**; no model training or evaluation is executed by this code.


In [ ]:
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator

class KaleTrainer(DefaultTrainer):

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):

        if output_folder is None:
            output_folder = os.path.join(
                cfg.OUTPUT_DIR,
                "inference"
            )

        return COCOEvaluator(
            dataset_name,
            output_dir=output_folder
        )

### Actual Training

In [ ]:
trainer = KaleTrainer(cfg)

trainer.resume_or_load(resume=False)

trainer.train()

### Final Model Evaluation on the Test Dataset

The trained Mask R-CNN model is evaluated on the independent **Purple Kale test dataset** using Detectron2's COCO-compatible evaluation pipeline. The final trained model checkpoint is loaded from the experiment output directory, and the test dataset is explicitly designated as the evaluation dataset.

A `COCOEvaluator` is initialized to compute standard COCO instance segmentation metrics, with the evaluation outputs stored in a dedicated test-results directory. A Detectron2 test data loader is then constructed to provide the test images and their corresponding ground-truth annotations to the evaluation pipeline.

The `inference_on_dataset` procedure performs inference using the trained model over the complete test dataset and passes the predictions to the COCO evaluator for quantitative assessment. The resulting metrics are collected and displayed, providing an objective measurement of the final model's instance segmentation performance on previously unseen data.


In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from detectron2.engine import DefaultPredictor

cfg.MODEL.WEIGHTS = os.path.join(
    cfg.OUTPUT_DIR,
    "model_final.pth"
)

cfg.DATASETS.TEST = ("kale_test",)

evaluator = COCOEvaluator(
    "kale_test",
    output_dir=os.path.join(cfg.OUTPUT_DIR, "test_results")
)

test_loader = build_detection_test_loader(
    cfg,
    "kale_test"
)

results = inference_on_dataset(
    trainer.model,
    test_loader,
    evaluator
)

print(results)